In [6]:
import os
from PIL import Image
import numpy as np

In [7]:
folder_normals = "./images-unity/baked"
out_folder = "./images-unity"

In [8]:
def normalize(v):
    norm = np.linalg.norm(v, axis=2, keepdims=True)
    norm = np.maximum(norm, 1e-8)
    return v / norm

def load_normal_image(im_name):
    path = os.path.join(folder_normals, im_name)
    img = Image.open(path).convert("RGB")
    rgb = np.array(img)
    mask_zeros = rgb.sum(axis=2) == 0
    # Conversion en vecteurs normaux
    n = np.array(rgb).astype(np.float32) / 255.0
    n = n * 2.0 - 1.0
    n = normalize(n)
    # Appliquer le masque
    n[mask_zeros] = (0.0, 0.0, 0.0)
    return n, mask_zeros[..., None]


In [9]:
test = np.array([-0,-0,0])
test_norm = normalize(test.reshape(1,1,3))
print(test_norm)
print(np.linalg.norm(test_norm))

[[[0. 0. 0.]]]
0.0


In [10]:
im_names = sorted([
        f for f in os.listdir(folder_normals)
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

# Charger la première image
img_mix, mask_zeros_mix = load_normal_image(im_names[0])

for im_name in im_names[1:]:
    img_next, mask_zeros_next = load_normal_image(im_name)

    img_mix = np.where(
        mask_zeros_next,
        img_mix,
        np.where(
            mask_zeros_mix,
            img_next,
            normalize(img_mix + img_next)
        )
    )
    mask_zeros_mix = mask_zeros_mix & mask_zeros_next


# Retour vers RGB

mask_zeros = img_mix == (0.0, 0.0, 0.0)
rgb_mix = (img_mix * 0.5 + 0.5) * 255.0
rgb_mix[mask_zeros] = 0
rgb_mix = np.clip(rgb_mix, 0, 255).astype(np.uint8)

# Sauvegarde
output_image = Image.fromarray(rgb_mix, "RGB")
output_image.show()

output_image.save(os.path.join(out_folder, "normal_merged.png"))
